# Confluence Data Center API サンプル

ローカル Docker で起動した Confluence Data Center に接続して API を試します。

**前提**
- Confluence の **プロフィール → Personal Access Tokens** でトークンを生成済みであること
- `.env` に `CONFLUENCE_DC_URL`, `CONFLUENCE_DC_PAT` を設定済みであること
- `docker compose restart jupyter` で環境変数を反映済みであること

In [1]:
from atlassian import Confluence
from dotenv import load_dotenv
import os
import json

load_dotenv()

# Jupyter コンテナから Confluence への接続
# - Docker ネットワーク内 (Jupyter → Confluence コンテナ): http://confluence:8090
# - ホストから直接アクセスする場合: http://localhost:8090
confluence_url = os.getenv("CONFLUENCE_DC_URL", "http://confluence:8090")

confluence = Confluence(
    url=confluence_url,
    token=os.getenv("CONFLUENCE_DC_PAT"),
)

print(f"接続先: {confluence_url}")

接続先: http://confluence:8090


## スペース一覧の取得

In [2]:
spaces = confluence.get_all_spaces(start=0, limit=25)
print(f"スペース数: {len(spaces.get('results', []))}")
print()
for s in spaces.get("results", []):
    print(f"  [{s['key']}] {s['name']} (type={s['type']})")

スペース数: 4

  [ds] Demonstration Space (type=global)
  [KS] KG Space (type=global)
  [~admin] admin (type=personal)
  [SPC] テスト (type=global)


## スペース内のページ一覧

In [7]:
# 最初のスペースを使用（変更する場合は SPACE_KEY を書き換えてください）
SPACE_KEY = spaces["results"][3]["key"]
print(f"対象スペース: {SPACE_KEY}")
print()

pages = confluence.get_all_pages_from_space(space=SPACE_KEY, start=0, limit=10)
print(f"ページ数: {len(pages)}")
for p in pages:
    print(f"  ID={p['id']}  {p['title']}")

対象スペース: SPC

ページ数: 6
  ID=131148  テスト
  ID=131158  2026-05-14のミーティングのメモ
  ID=131171  2026-05-14 ミーティング議事録
  ID=131172  ミーティング議事録
  ID=1605635  マクロテスト
  ID=1605644  アンカーリンクテスト


## ページ本文の取得

In [8]:
page_id = pages[4]["id"]
page = confluence.get_page_by_id(
    page_id,
    expand="body.storage,version,history,ancestors"
)

print("Title   :", page["title"])
print("Version :", page["version"]["number"])
print("Creator :", page["history"]["createdBy"]["displayName"])
print("Created :", page["history"]["createdDate"])
print()
print("=== Body (storage format, first 1000 chars) ===")
print(page["body"]["storage"]["value"][:1000])

Title   : マクロテスト
Version : 5
Creator : admin
Created : 2026-05-15T06:59:31.605Z

=== Body (storage format, first 1000 chars) ===
<p><ac:structured-macro ac:name="anchor" ac:schema-version="1" ac:macro-id="ff212348-59f4-4a91-bd6d-326f7f69051c"><ac:parameter ac:name="">macro-test-top</ac:parameter></ac:structured-macro></p><p><br /></p><p><br /></p><ac:structured-macro ac:name="code" ac:schema-version="1" ac:macro-id="6320a990-0d57-436f-85a9-b5b1ba487bd8"><ac:parameter ac:name="language">python</ac:parameter><ac:plain-text-body><![CDATA[if __name___ == "__main__":
	print("hello world.")]]></ac:plain-text-body></ac:structured-macro><p><br /></p>


## CQL 検索

In [5]:
# CQL (Confluence Query Language) で最近更新されたページを検索
cql_query = f'space="{SPACE_KEY}" AND type=page ORDER BY lastmodified DESC'
results = confluence.cql(cql_query, limit=5)

print(f"CQL: {cql_query}")
print(f"ヒット数: {results.get('totalSize', 0)}")
print()
for r in results.get("results", []):
    title = r.get("title", "(no title)")
    # CQL 結果は _links が content 配下にある場合と url フィールドがある場合がある
    webui = (
        r.get("url")
        or r.get("content", {}).get("_links", {}).get("webui", "")
    )
    url = confluence_url + webui if webui and not webui.startswith("http") else webui
    print(f"  {title}")
    print(f"    {url}")

CQL: space="KS" AND type=page ORDER BY lastmodified DESC
ヒット数: 3

  How To サンプル
    http://confluence:8090/spaces/KS/pages/1081357/How+To+%E3%82%B5%E3%83%B3%E3%83%97%E3%83%AB
  How To記事
    http://confluence:8090/spaces/KS/pages/1081358/How+To%E8%A8%98%E4%BA%8B
  KG Space
    http://confluence:8090/spaces/KS/pages/1081353/KG+Space


## 添付ファイル一覧の取得

In [6]:
attachments = confluence.get_attachments_from_content(page_id=page_id)
att_list = attachments.get("results", [])
print(f"添付ファイル数: {len(att_list)}")
for a in att_list:
    print(f"  {a['title']} ({a['metadata']['mediaType']}, {a['extensions']['fileSize']} bytes)")

添付ファイル数: 0
